# Chapter 08: Non-parametric Methods

Source span: printed pages 145-158; PDF pages 161-174. The textbook PDF is used here only for orientation: chapter structure, terminology, and concept order. The prose, examples, diagrams, computations, and artifacts in this notebook are original course material.

## Chapter Question

Distribution-free circular inference through symmetry, ranks, uniform scores, two-sample comparisons, runs, and q-sample tests. The question is how to make that statement visible and testable. Directional statistics is not ordinary Euclidean statistics with trigonometric decoration. Its sample spaces carry identifications: angles wrap, axes ignore sign, spherical data rotate, frames preserve orthogonality, and shapes forget translation, scale, and rotation. Each section below converts those identifications into a computational object and then into an artifact the reader can inspect.

The chapter avoids parametric assumptions by moving ranks and runs onto the circle. Uniform scores, two-sample circular CDFs, and runs parity are used as direct checks that the procedure respects periodic order.

A useful reading habit is to pause before every formula and ask what transformation should leave it unchanged. If shifting the zero angle, rotating the sphere, changing the basis of a subspace, or translating a landmark configuration changes the scientific answer, the calculation is not yet respecting the sample space. The notebook's final checks are built around that habit.


## Translation Guide

                Concepts translated in this lesson:

                - rank ideas survive on the circle when ranks are converted into uniform scores
- symmetry tests ask whether reflected directions match
- two-sample tests compare circular empirical distributions
- runs and q-sample methods reveal distributional differences without a von Mises model

                The computational translation is intentionally concrete. Observations are represented as arrays only after the relevant geometry has been named. Circular observations are wrapped before differences are computed. Spherical observations are normalized and summarized by vector resultants or matrix spectra. Rotations, frames, and subspaces are checked by orthogonality, determinant, or idempotence identities. Shapes are centered, scaled, and aligned before comparison.

                This guide matters because the same numerical array can mean different things. A pair of columns might be two ordinary variables, sine-cosine coordinates for a circle, a two-frame on a Stiefel manifold, or a landmark configuration after centering. The formulas in the course are useful only when paired with the right interpretation.


In [ ]:
from pathlib import Path
import sys

def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "Directional Statistics" and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Could not locate Directional Statistics course root")

BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ENTRY = {'kind': 'chapter', 'number': 8, 'label': 'Chapter 08', 'title': 'Non-parametric Methods', 'folder': 'chapter-08-non-parametric-methods', 'notebook': '08-non-parametric-methods.ipynb', 'part': 'part-01-circular-statistics', 'topic': 'chapter-08', 'family': 'circular', 'printed': '145-158', 'pdf': '161-174', 'focus': 'Distribution-free circular inference through symmetry, ranks, uniform scores, two-sample comparisons, runs, and q-sample tests.', 'concepts': ['rank ideas survive on the circle when ranks are converted into uniform scores', 'symmetry tests ask whether reflected directions match', 'two-sample tests compare circular empirical distributions', 'runs and q-sample methods reveal distributional differences without a von Mises model'], 'visuals': ['symmetry sign and Wilcoxon display', 'uniform-score construction', 'two-sample circular CDF overlays', 'circular runs diagram'], 'checks': ['tests are invariant under rotation', 'ties are reported and handled consistently', 'runs count is even on a circle', 'uniform-score statistic detects location and dispersion differences']}
TOPIC = ENTRY["topic"]
print(f"Course root: {BOOK_ROOT}")
print(f"Working topic: {TOPIC}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.topic_visuals import make_topic_interactive_figure, make_topic_static_figure, topic_numeric_checks
from utils.validation import assert_artifacts

np.set_printoptions(precision=4, suppress=True)
source_span = {"printed": ENTRY["printed"], "pdf": ENTRY["pdf"]}
source_span


## Visual Storyboard

                The visual sequence for this notebook is:

                - symmetry sign and Wilcoxon display
- uniform-score construction
- two-sample circular CDF overlays
- circular runs diagram

                The static figure is the durable teaching artifact. It is designed to be understandable in a rendered notebook, a static export, or a file browser. The interactive HTML artifact is self-contained and book-local; it lets the reader rotate or inspect the geometry without relying on the PDF or a network connection. Together the two artifacts should answer two different questions: what invariant is being measured, and how does the picture change when the sample or model changes?

                The displayed figure is not a screenshot from the source. It is generated from small synthetic examples chosen to expose the chapter's geometry. Synthetic data keeps the lesson reproducible and avoids copying long tables. The examples are small by design: they make it easy to predict what should happen before the code runs.


In [ ]:
fig, static_diagnostics = make_topic_static_figure(ENTRY)
png_path = save_matplotlib(fig, TOPIC, "core", "concept-diagnostic.png")
plt.close(fig)
display_artifact(png_path, width=860)
static_diagnostics


## Worked Lab

                The lab cell below builds the chapter-specific artifact and records the numerical diagnostics that make the visual auditable. Read the numbers beside the figure rather than treating them as invisible bookkeeping. A resultant length, an integration residual, an idempotence norm, or a Procrustes invariance error is a compact statement about the geometry in the picture.

                Expected checks for this chapter:

                - tests are invariant under rotation
- ties are reported and handled consistently
- runs count is even on a circle
- uniform-score statistic detects location and dispersion differences

                After running the visual cell, change one parameter in the helper call and predict the effect. Increase concentration and the cluster should tighten. Rotate all angles and origin-invariant tests should not change. Add axial symmetry and a mean-direction diagnostic should become less informative. Translate a shape and Procrustes comparisons should be stable. These small perturbations are the quickest way to find a hidden linear assumption.


In [ ]:
interactive_fig = make_topic_interactive_figure(ENTRY)
html_path = save_plotly_html(interactive_fig, TOPIC, "interactive", "exploration.html", include_plotlyjs=True)
display_artifact(html_path, width="100%", height=520)


In [ ]:
numeric_checks = topic_numeric_checks(ENTRY, static_diagnostics)
# Every chapter should contribute at least one boolean mathematical invariant
# in addition to artifact existence checks.
boolean_checks = [value for key, value in numeric_checks.items() if isinstance(value, bool)]
assert boolean_checks, "no topic-specific boolean checks were recorded"
assert all(boolean_checks), numeric_checks
checks_path = save_json(numeric_checks, TOPIC, "checks", "numeric-checks.json")
numeric_checks


## Interpretation Notes

The key definitions in this notebook should be read operationally. The phrase `Non-parametric Methods` names a family of decisions about representation: which coordinates are safe to use, which transformations are nuisance transformations, and which numerical summaries survive those transformations. The concept list above is the local contract for those decisions. If a learner can explain why each listed concept is invariant, equivariant, or intentionally coordinate-dependent, then the formulas have become usable rather than memorized.

The visual artifact is also an argument. It chooses one small scene where the chapter's main failure mode is visible: a bad cut point, a misleading linear average, a density that must integrate around a closed loop, a matrix that must remain orthogonal, or a shape comparison that must ignore translation and scale. In later applications the data may be larger and noisier, but the same small-scene logic applies. Start with a display that makes the invariant visible, then scale the computation.

The saved JSON checks are deliberately plain. They are meant to be read by a person and by an audit script. A check such as `A_kappa_matches_R`, `unit_vectors_on_sphere`, `grassmann_projection_idempotent`, or `procrustes_ignores_translation` is a compact record of what would have gone wrong if the sample space had been flattened too early. When extending this notebook, add checks of the same kind before adding visual flourish.

Finally, notice that the artifacts are book-local. The notebook does not depend on a PDF crop, an external web service, or a hidden interactive session. This matters pedagogically: the reader can inspect the figure, rerun the cells, compare the JSON diagnostics, and then use the same helper functions in a new data analysis. That is the course standard for a standalone visualization-first chapter.

Before moving on, use this reader checklist. Identify the observation type in one sentence. Point to the panel or interactive scene where that type is visible. Name the statistic being computed and the transformation it should respect. Then open the final sanity JSON and find the value that supports that claim. If any of those four steps feels vague, the notebook should be extended at that point rather than treated as complete. The aim is not to make every chapter long; the aim is to make every chapter inspectable, portable, and honest about the geometry it uses.

This short checklist is also the acceptance test for future revisions.


## Takeaways

The central takeaway is that non-parametric methods is a geometry-first topic. The statistic, the visualization, and the final sanity check should all refer to the same invariant. If the artifact shows a rose plot but the chapter is about likelihood, the artifact is wrong; if a final JSON file only says that files exist, the check is too weak. This notebook therefore saves both concept-named artifacts and chapter-specific numerical checks.

Keep three habits for later notebooks. First, name the sample space before choosing a coordinate system. Second, draw the invariant or failure mode directly. Third, keep a small numerical check beside every visual claim. That discipline is what makes the course standalone: the reader does not need the textbook open to understand what is being measured, why the geometry matters, and how to verify the computation.


In [ ]:
final_sanity = {
    "artifacts": assert_artifacts([png_path, html_path, checks_path], min_bytes=100),
    "topic_checks": numeric_checks,
    "standalone_contract": "original prose, generated visuals, executable checks",
    "pdf_used_for": "source orientation only",
}
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 100
final_sanity
